# データ保存

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# 電子

In [ ]:
import pyspedas as psp
import ergpyspedas.erg as ergpy
import pytplot as pt
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import xarray as xr

pt.del_data('*')

trange = ['2022-09-01/22:25', '2022-09-01/23:15']

ergpy.lepe(trange=trange, level='l2', datatype='3dflux')
ergpy.lepe(trange=trange, level='l2', datatype='omniflux')
ergpy.mgf(trange=trange, level='l2', datatype='64hz', coord='dsi')
ergpy.orb(trange=trange, level='l2')

In [ ]:
background_time_sec = 100 #[sec]

In [ ]:
import xarray as xr
import numpy as np

time_range_T    = [trange[0].replace('/', 'T'), trange[1].replace('/', 'T')]

B64_data_dsi    = psp.get_data('erg_mgf_l2_mag_64hz_dsi', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
B64_data_dsi_quality_flag   = psp.get_data('erg_mgf_l2_quality_64hz', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

qf_B, B64 = xr.align(B64_data_dsi_quality_flag[:, 3], B64_data_dsi, join='inner')

B64_data_dsi_qf = xr.where(qf_B <= 21, B64, np.nan)

ds_B64_dsi  = xr.Dataset({
    'B64_dsi_x':    B64_data_dsi_qf[:, 0],
    'B64_dsi_y':    B64_data_dsi_qf[:, 1],
    'B64_dsi_z':    B64_data_dsi_qf[:, 2]
})

ds_B64_dsi  = ds_B64_dsi.dropna(dim='time', how='all')

In [ ]:
da_mgf_spin_phase_deg           = psp.get_data('erg_mgf_l2_spin_phase_64hz', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
da_mgf_spin_phase_deg_interp    = da_mgf_spin_phase_deg.interp(time=ds_B64_dsi.time)
da_mgf_spin_phase_rad_interp    = np.deg2rad(da_mgf_spin_phase_deg_interp)

In [ ]:
import sys, importlib
importlib.invalidate_caches()
os.chdir('..')
print(os.getcwd())
import module_handmade.erg_mgf_spintone_rm as emsr
importlib.reload(emsr)
os.chdir('./KAW_observation')
print(os.getcwd())

B_clean_ndarray, B_spt_ndarray, params  = emsr.remove_spintone_3comp(
    time=ds_B64_dsi.time.values,
    Bx=ds_B64_dsi['B64_dsi_x'].values,
    By=ds_B64_dsi['B64_dsi_y'].values,
    Bz=ds_B64_dsi['B64_dsi_z'].values,
    phase_rad=da_mgf_spin_phase_rad_interp.values,
    min_points=64.*3./2.
)

ds_B64_dsi_spt      = xr.Dataset({
    'B64_dsi_x_spt':    ('time', B_spt_ndarray[:, 0]),
    'B64_dsi_y_spt':    ('time', B_spt_ndarray[:, 1]),
    'B64_dsi_z_spt':    ('time', B_spt_ndarray[:, 2])
    }, coords={'time': ds_B64_dsi.time.values})

ds_B64_dsi_clean    = xr.Dataset({
    'B64_dsi_x_clean':    ('time', B_clean_ndarray[:, 0]),
    'B64_dsi_y_clean':    ('time', B_clean_ndarray[:, 1]),
    'B64_dsi_z_clean':    ('time', B_clean_ndarray[:, 2])
    }, coords={'time': ds_B64_dsi.time.values})

print(ds_B64_dsi_spt)
print(ds_B64_dsi_clean)

In [ ]:
time_width_B64          = (ds_B64_dsi_clean.time[10] - ds_B64_dsi_clean.time[9]) / np.timedelta64(1, 's')
ds_B_background         = ds_B64_dsi_clean.rolling(time=int(background_time_sec / time_width_B64), center=True).mean('time')

da_B_background         = ds_B_background.to_dataarray(dim='v_dim').T.dropna(dim='time', how='any')

print(da_B_background)

In [ ]:
psp.store_data('erg_mgf_l2_mag_64hz_background_dsi', data={'x': da_B_background.time, 'y': da_B_background.data})

In [ ]:
FEDU_energy_list = np.sort(np.unique(psp.get_data('erg_lepe_l2_3dflux_FEDU', xarray=True).v1[0, :].data))
print(FEDU_energy_list)

In [ ]:
FEDU_energy_center  = FEDU_energy_list

log_center = np.log10(FEDU_energy_center)
log_edges = np.zeros(len(FEDU_energy_center) + 1)
log_edges[1:-1] = 0.5 * (log_center[:-1] + log_center[1:])
log_edges[0] = log_center[0] + (log_center[0] - log_edges[1])
log_edges[-1] = log_center[-1] - (log_edges[-2] - log_center[-1])

FEDU_energy_grid = 10**log_edges

print(FEDU_energy_grid)

In [ ]:
for iel, el in enumerate(FEDU_energy_center):

    psp.projects.erg.erg_lep_part_products(
        'erg_lepe_l2_3dflux_FEDU',
        outputs=['pa'],
        pitch=[0, 180],
        energy=[FEDU_energy_grid[iel], FEDU_energy_grid[iel+1]],
        mag_name='erg_mgf_l2_mag_64hz_background_dsi',
        pos_name='erg_orb_l2_pos_sm',
        suffix=f'_{iel}'
    )

In [ ]:
mpl.rcParams['font.size'] = 20

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import pytplot as pt
import matplotlib.ticker as mticker

def omniflux_plot_Arase(inst_name, fig, ax, cax, dq, vmin=None, vmax=None, energy_min=None, energy_max=None, ytitle=None, ztitle=None):
    # 取り出し（全て (T, E) 形状）
    flux = np.asarray(dq.data, dtype=float)           # (T, E)
    if inst_name == 'lepe':
        E    = np.asarray(dq.spec_bins, dtype=float)      # (T, E) 変動エネルギー
    elif inst_name == 'lepi':
        E    = np.asarray(dq.spec_bins, dtype=float)      # (E) 変動エネルギー
        E    = E * 1E3                                     # keV → eV
        E    = np.broadcast_to(E[None, :], flux.shape)      # (T, E)
    t    = np.asarray(dq.time.values)                 # (T,)

    print("before:", flux.shape, E.shape)

    # 列方向で有限なエネルギーチャンネルだけ残す
    if E.ndim == 2:
        good_e = np.all(np.isfinite(E), axis=0)
    elif E.ndim == 1:
        good_e = np.isfinite(E)
    else:
        raise ValueError(f"unexpected E.ndim={E.ndim}, E.shape={E.shape}")

    if not np.any(good_e):
        raise ValueError("有効なエネルギーチャンネルがありません。")

    flux = flux[:, good_e]
    if E.ndim == 2:
        E = E[:, good_e]
    else:
        E = np.broadcast_to(E[None, good_e], flux.shape)

    print("after :", flux.shape, E.shape)

    # LogNorm 用
    flux = flux.copy()
    flux[~np.isfinite(flux)] = np.nan
    flux[flux <= 0] = np.nan

    if np.all(~np.isfinite(flux)):
        raise ValueError("有効な（>0）フラックスがありません。")

    if vmin == None:
        vmin = np.nanmin(flux)
    if vmax == None:
        vmax = np.nanmax(flux)
    if not (vmin < vmax):
        vmax = vmin * 1.0001
    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    print(vmin, vmax)

    flux[np.isnan(flux)] = 1E-99

    # ---- 時間メッシュ生成（(T,E)）----
    Tmesh = np.broadcast_to(t[:, None], E.shape)
    print(Tmesh.shape)

    # ---- 描画 ----
    mesh = ax.pcolormesh(Tmesh, E, flux, norm=norm, cmap=cm.turbo, shading='nearest')
    cb = fig.colorbar(mesh, cax=cax)
    cb.set_label("")  # ラベル消す
    #if ztitle:
    #    ax.text(0.85, 0.15, ztitle,
    #            transform=ax.transAxes, ha='center', va='center', color='k',
    #            bbox=dict(facecolor='white', alpha=0.5, boxstyle='round,pad=0.25'))

    ax.set_yscale('log')
    ax.set_ylabel(ytitle)
    ax.grid(which='both', alpha=0.5)
    ax.minorticks_on()

    if energy_min != None:
        ax.set_ylim(ymin=energy_min)
    if energy_max != None:
        ax.set_ylim(ymax=energy_max)

    # 時刻目盛り
    loc = mdates.AutoDateLocator()
    fmt = mdates.ConciseDateFormatter(loc)
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(fmt)

    return fig, ax, cax

def pa_plot_Arase(fig, ax, cax, dq, vmin=None, vmax=None, ytitle=None, ztitle=None):
    # 取り出し（全て (T, E) 形状）
    flux = np.asarray(dq.data, dtype=float)           # (T, E)
    PA    = np.asarray(dq.spec_bins, dtype=float)     # (T, E)
    t    = np.asarray(dq.time.values)                 # (T,)

    # pcolormeshのX/Yは非有限NG → 列方向で全部有限なチャンネルだけ残す
    good_PA = np.all(~np.isnan(PA), axis=0)
    if not np.all(good_PA):
        flux = flux[:, good_PA]
        PA    = PA[:,    good_PA]

    # ---- 値の前処理（LogNorm 用）----
    flux[~np.isfinite(flux)] = np.nan
    flux[flux <= 0] = np.nan
    if np.all(~np.isfinite(flux)):
        raise ValueError("有効な（>0）フラックスがありません。")

    if vmin == None:
        vmin = np.nanmin(flux)
    if vmax == None:
        vmax = np.nanmax(flux)
    if not (vmin < vmax):
        vmax = vmin * 1.0001
    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    print(vmin, vmax)

    flux[np.isnan(flux)] = 1E-99

    # ---- 時間メッシュ生成（(T,E)）----
    Tmesh = np.broadcast_to(t[:, None], PA.shape)

    print(flux.shape)
    print(PA.shape)
    print(Tmesh.shape)

    # ---- 描画 ----
    mesh = ax.pcolormesh(Tmesh, PA, flux, norm=norm, cmap=cm.turbo, shading='nearest')
    cb = fig.colorbar(mesh, cax=cax)

    cb.set_label(ztitle)  # ラベル消す
    #if ztitle:
    #    ax.text(0.85, 0.5, ztitle,
    #            transform=ax.transAxes, ha='center', va='center', color='k',
    #            bbox=dict(facecolor='white', alpha=0.5, boxstyle='round,pad=0.25'))

    ax.set_ylabel(ytitle)
    ax.grid(which='both', alpha=0.5)
    ax.minorticks_on()

    ax.set_ylim(ymax=180, ymin=0)
    ax.set_yticks(np.arange(0, 181, 45))

    # 時刻目盛り
    loc = mdates.AutoDateLocator()
    fmt = mdates.ConciseDateFormatter(loc)
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(fmt)

    return fig, ax, cax

In [ ]:
#out_dir = f'/mnt/j/KAW_observation/LEP-e_pitch_angle_each_time/20220901/'
#
#for iel, el in enumerate(FEDU_energy_center):
#
#    fig = plt.figure(figsize=(15, 3))
#    gs = fig.add_gridspec(1, 2, width_ratios=[1, 0.025], wspace=0.05)
#    ax = fig.add_subplot(gs[0, 0])
#    cax = fig.add_subplot(gs[0, 1])
#    try:
#        da = psp.get_data(f'erg_lepe_l2_3dflux_FEDU_pa_{iel}', xarray=True).sel(time=slice(*trange))    #'#/s/$cm^{2}$/str/eV
#    except:
#        continue
#    da_vmin = np.nanpercentile(da.data, 50)
#    da_vmax = np.nanpercentile(da.data, 99)
#    ytitle = f'{el:.0f} eV\n [deg]'
#    ztitle = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
#
#    pa_plot_Arase(fig, ax, cax, da, da_vmin, da_vmax, ytitle, ztitle)
#
#    fig.tight_layout()
#    fig.savefig(os.path.join(out_dir, f'electron_PA_{iel}.png'), dpi=200, bbox_inches='tight')
#    plt.close(fig)

In [ ]:
#fig = plt.figure(figsize=(15, 3))
#gs = fig.add_gridspec(1, 2, width_ratios=[1, 0.025], wspace=0.05)
#ax = fig.add_subplot(gs[0, 0])
#cax = fig.add_subplot(gs[0, 1])
#
#da = psp.get_data(f'erg_lepe_l2_omniflux_FEDO', xarray=True).sel(time=slice(*trange))
#da_vmin = np.nanpercentile(da.data, 50)
#da_vmax = np.nanpercentile(da.data, 99)
#ytitle = f'omniflux\n [eV]'
#ztitle = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
#
#omniflux_plot_Arase('lepe',fig, ax, cax, da, da_vmin, da_vmax, 5E1, 1E4, ytitle, ztitle)
#fig.tight_layout()
#fig.savefig(os.path.join(out_dir, f'electron_omniflux.png'), dpi=200, bbox_inches='tight')
#plt.close(fig)

# $\mathrm{H}^{+}$

In [ ]:
ergpy.lepi(trange=trange, level='l2', datatype='3dflux')
ergpy.lepi(trange=trange, level='l2', datatype='omniflux')

In [ ]:
FPDU_energy_list = np.sort(np.unique(psp.get_data('erg_lepi_l2_3dflux_FPDU', xarray=True).v1.data)) * 1E3
print(FPDU_energy_list)

In [ ]:
FPDU_energy_center  = FPDU_energy_list

log_center = np.log10(FPDU_energy_center)
log_edges = np.zeros(len(FPDU_energy_center) + 1)
log_edges[1:-1] = 0.5 * (log_center[:-1] + log_center[1:])
log_edges[0] = log_center[0] + (log_center[0] - log_edges[1])
log_edges[-1] = log_center[-1] - (log_edges[-2] - log_center[-1])

FPDU_energy_grid = 10**log_edges

print(FPDU_energy_grid)

In [ ]:
for iel, el in enumerate(FPDU_energy_center):

    psp.projects.erg.erg_lep_part_products(
        'erg_lepi_l2_3dflux_FPDU',
        outputs=['pa'],
        pitch=[0, 180],
        energy=[FPDU_energy_grid[iel], FPDU_energy_grid[iel+1]],
        mag_name='erg_mgf_l2_mag_64hz_background_dsi',
        pos_name='erg_orb_l2_pos_sm',
        suffix=f'_{iel}'
    )

In [ ]:
mpl.rcParams['font.size'] = 20

In [ ]:
out_dir = f'/mnt/j/KAW_observation/LEP-i_P_pitch_angle_each_time/20220901/'

for iel, el in enumerate(FPDU_energy_center):

    fig = plt.figure(figsize=(15, 3))
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 0.025], wspace=0.05)
    ax = fig.add_subplot(gs[0, 0])
    cax = fig.add_subplot(gs[0, 1])
    try:
        da = psp.get_data(f'erg_lepi_l2_3dflux_FPDU_pa_{iel}', xarray=True).sel(time=slice(*trange))    #'#/s/$cm^{2}$/str/eV
    except:
        continue
    da_vmin = np.nanpercentile(da.data, 50)
    da_vmax = np.nanpercentile(da.data, 99)
    if da_vmin < da_vmax * 1E-2:
        da_vmin = da_vmax * 1E-2
    print(da_vmax)
    if da_vmax < 1E-10 or np.isnan(da_vmax):
        continue
    ytitle = f'{el:.0f} eV\n [deg]'
    ztitle = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'

    pa_plot_Arase(fig, ax, cax, da, da_vmin, da_vmax, ytitle, ztitle)

    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, f'proton_PA_{iel}.png'), dpi=200, bbox_inches='tight')
    plt.close(fig)

In [ ]:
fig = plt.figure(figsize=(15, 3))
gs = fig.add_gridspec(1, 2, width_ratios=[1, 0.025], wspace=0.05)
ax = fig.add_subplot(gs[0, 0])
cax = fig.add_subplot(gs[0, 1])

da = psp.get_data(f'erg_lepi_l2_omniflux_FPDO', xarray=True).sel(time=slice(*trange)) * 1E-3
da_vmin = np.nanpercentile(da.data, 50)
da_vmax = np.nanpercentile(da.data, 99)
ytitle = f'omniflux\n [eV]'
ztitle = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'

omniflux_plot_Arase('lepi',fig, ax, cax, da, da_vmin, da_vmax, 5E1, 1E4, ytitle, ztitle)
fig.tight_layout()
fig.savefig(os.path.join(out_dir, f'proton_omniflux.png'), dpi=200, bbox_inches='tight')
plt.close(fig)

# $\mathrm{He}^{+}$

In [ ]:
FHEDU_energy_list = np.sort(np.unique(psp.get_data('erg_lepi_l2_3dflux_FHEDU', xarray=True).v1.data)) * 1E3
print(FHEDU_energy_list)

In [ ]:
FHEDU_energy_center  = FHEDU_energy_list

log_center = np.log10(FHEDU_energy_center)
log_edges = np.zeros(len(FHEDU_energy_center) + 1)
log_edges[1:-1] = 0.5 * (log_center[:-1] + log_center[1:])
log_edges[0] = log_center[0] + (log_center[0] - log_edges[1])
log_edges[-1] = log_center[-1] - (log_edges[-2] - log_center[-1])

FHEDU_energy_grid = 10**log_edges

print(FHEDU_energy_grid)

In [ ]:
for iel, el in enumerate(FHEDU_energy_center):

    psp.projects.erg.erg_lep_part_products(
        'erg_lepi_l2_3dflux_FHEDU',
        outputs=['pa'],
        pitch=[0, 180],
        energy=[FHEDU_energy_grid[iel], FHEDU_energy_grid[iel+1]],
        mag_name='erg_mgf_l2_mag_64hz_background_dsi',
        pos_name='erg_orb_l2_pos_sm',
        suffix=f'_{iel}'
    )

In [ ]:
mpl.rcParams['font.size'] = 20

In [ ]:
out_dir = f'/mnt/j/KAW_observation/LEP-i_HE_pitch_angle_each_time/20220901/'

for iel, el in enumerate(FHEDU_energy_center):

    fig = plt.figure(figsize=(15, 3))
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 0.025], wspace=0.05)
    ax = fig.add_subplot(gs[0, 0])
    cax = fig.add_subplot(gs[0, 1])
    try:
        da = psp.get_data(f'erg_lepi_l2_3dflux_FHEDU_pa_{iel}', xarray=True).sel(time=slice(*trange))    #'#/s/$cm^{2}$/str/eV
    except:
        continue
    da_vmin = np.nanpercentile(da.data, 97.5)
    da_vmax = np.nanpercentile(da.data, 99)
    if da_vmin < da_vmax * 1E-2:
        da_vmin = da_vmax * 1E-2
    print(da_vmax)
    if da_vmax < 1E-10 or np.isnan(da_vmax):
        continue
    ytitle = f'{el:.0f} eV\n [deg]'
    ztitle = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'

    pa_plot_Arase(fig, ax, cax, da, da_vmin, da_vmax, ytitle, ztitle)

    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, f'helium_PA_{iel}.png'), dpi=200, bbox_inches='tight')
    plt.close(fig)

In [ ]:
fig = plt.figure(figsize=(15, 3))
gs = fig.add_gridspec(1, 2, width_ratios=[1, 0.025], wspace=0.05)
ax = fig.add_subplot(gs[0, 0])
cax = fig.add_subplot(gs[0, 1])

da = psp.get_data(f'erg_lepi_l2_omniflux_FHEDO', xarray=True).sel(time=slice(*trange)) * 1E-3
da_vmin = np.nanpercentile(da.data, 90)
da_vmax = np.nanpercentile(da.data, 99)
ytitle = f'omniflux\n [eV]'
ztitle = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'

omniflux_plot_Arase('lepi',fig, ax, cax, da, da_vmin, da_vmax, 5E1, 1E4, ytitle, ztitle)
fig.tight_layout()
fig.savefig(os.path.join(out_dir, f'helium_omniflux.png'), dpi=200, bbox_inches='tight')
plt.close(fig)

# $\mathrm{O}^{+}$

In [ ]:
FODU_energy_list = np.sort(np.unique(psp.get_data('erg_lepi_l2_3dflux_FODU', xarray=True).v1.data)) * 1E3
print(FODU_energy_list)

In [ ]:
FODU_energy_center  = FODU_energy_list

log_center = np.log10(FODU_energy_center)
log_edges = np.zeros(len(FODU_energy_center) + 1)
log_edges[1:-1] = 0.5 * (log_center[:-1] + log_center[1:])
log_edges[0] = log_center[0] + (log_center[0] - log_edges[1])
log_edges[-1] = log_center[-1] - (log_edges[-2] - log_center[-1])

FODU_energy_grid = 10**log_edges

print(FODU_energy_grid)

In [ ]:
for iel, el in enumerate(FODU_energy_center):

    psp.projects.erg.erg_lep_part_products(
        'erg_lepi_l2_3dflux_FODU',
        outputs=['pa'],
        pitch=[0, 180],
        energy=[FODU_energy_grid[iel], FODU_energy_grid[iel+1]],
        mag_name='erg_mgf_l2_mag_64hz_background_dsi',
        pos_name='erg_orb_l2_pos_sm',
        suffix=f'_{iel}'
    )

In [ ]:
mpl.rcParams['font.size'] = 20

In [ ]:
out_dir = f'/mnt/j/KAW_observation/LEP-i_O_pitch_angle_each_time/20220901/'

for iel, el in enumerate(FODU_energy_center):

    fig = plt.figure(figsize=(15, 3))
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 0.025], wspace=0.05)
    ax = fig.add_subplot(gs[0, 0])
    cax = fig.add_subplot(gs[0, 1])
    try:
        da = psp.get_data(f'erg_lepi_l2_3dflux_FODU_pa_{iel}', xarray=True).sel(time=slice(*trange))    #'#/s/$cm^{2}$/str/eV
    except:
        continue
    da_vmin = np.nanpercentile(da.data, 97.5)
    da_vmax = np.nanpercentile(da.data, 99)
    if da_vmin < da_vmax * 1E-2:
        da_vmin = da_vmax * 1E-2
    print(da_vmax)
    if da_vmax < 1E-10 or np.isnan(da_vmax):
        continue
    ytitle = f'{el:.0f} eV\n [deg]'
    ztitle = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'

    pa_plot_Arase(fig, ax, cax, da, da_vmin, da_vmax, ytitle, ztitle)

    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, f'oxygen_PA_{iel}.png'), dpi=200, bbox_inches='tight')
    plt.close(fig)

In [ ]:
fig = plt.figure(figsize=(15, 3))
gs = fig.add_gridspec(1, 2, width_ratios=[1, 0.025], wspace=0.05)
ax = fig.add_subplot(gs[0, 0])
cax = fig.add_subplot(gs[0, 1])

da = psp.get_data(f'erg_lepi_l2_omniflux_FODO', xarray=True).sel(time=slice(*trange)) * 1E-3
da_vmin = np.nanpercentile(da.data, 95)
da_vmax = np.nanpercentile(da.data, 99)
ytitle = f'omniflux\n [eV]'
ztitle = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'

omniflux_plot_Arase('lepi',fig, ax, cax, da, da_vmin, da_vmax, 5E1, 1E4, ytitle, ztitle)
fig.tight_layout()
fig.savefig(os.path.join(out_dir, f'oxygen_omniflux.png'), dpi=200, bbox_inches='tight')
plt.close(fig)